
# Regularization — Ridge · Lasso · ElasticNet 을 코드로 보다

> 본 노트북의 목적은 핸드아웃 `Regularization_handout.md` 의 다섯 단계를 코드로 직접 재현하여,  
> **정규화가 OLS 의 한계 (다중공선성·과적합) 를 어떻게 보완하는가** 를 그래프로 \"보는\" 것이다.

**선행자료**: `OLS_notebook.ipynb` (행렬형 OLS 정규방정식), `PCA_notebook.ipynb` (표준화의 필요성).

**오늘의 다섯 단계**

1. 자료 준비 — 상관 강한 합성 자료, 표준화.
2. OLS 적합 — 일부 계수가 불안정하게 큰 모습.
3. Ridge — 정규화 경로 (모든 계수가 부드럽게 0 으로).
4. Lasso — 정규화 경로 (일부 계수가 정확히 0 으로).
5. 교차검증으로 최적 $\alpha$ 선택 → 시험 자료 성능 비교.



## 0.  임포트와 합성 자료 만들기

20 개 설명변수 중 다섯 개만 실제로 $y$ 에 영향을 주고, 나머지는 잡음 변수이다. 일부 변수는 서로 강하게 상관시키도록 설계한다 — 다중공선성의 효과를 보기 위함이다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

rng = np.random.default_rng(seed=20260515)

n, p = 80, 20
beta_true = np.zeros(p)
beta_true[:5] = [3.0, -2.0, 1.5, -1.0, 0.5]   # 의미 있는 변수 5 개

# 강한 상관을 만들기 위해 일부 변수를 다른 변수의 선형결합 + 잡음으로 만든다
Z = rng.normal(0, 1, (n, p))
Z[:, 1] = 0.95 * Z[:, 0] + 0.10 * rng.normal(0, 1, n)   # X1 ≈ X0
Z[:, 6] = 0.90 * Z[:, 5] + 0.15 * rng.normal(0, 1, n)   # X6 ≈ X5

X = Z
y = X @ beta_true + rng.normal(0, 1.0, n)               # 잡음 σ=1

print(f"X: shape={X.shape}")
print(f"y: shape={y.shape}")
print(f"참 베타 (앞 8개): {np.round(beta_true[:8], 2)}")
print(f"  ⇒ 나머지 {p-5} 개 변수는 잡음 (참 계수 = 0)")

# 훈련/시험 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)
print(f"\n훈련: {X_train.shape},  시험: {X_test.shape}")

# 표준화 — 훈련에서 학습한 평균/표준편차를 시험에 적용
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"\n표준화 후 훈련 평균: {X_train_s.mean(axis=0)[:3].round(3)}, ...")
print(f"표준화 후 훈련 표준편차: {X_train_s.std(axis=0)[:3].round(3)}, ...")



## 1.  OLS 적합 — 다중공선성의 효과를 직접 보다

OLS 를 그대로 적용하면 강한 상관을 가진 변수의 계수가 \"서로 상쇄하면서 크게 출렁이는\" 모양을 볼 수 있다. 시험 자료의 예측 오차도 크다.


In [ ]:
ols = LinearRegression().fit(X_train_s, y_train)
beta_ols = ols.coef_

y_pred_ols = ols.predict(X_test_s)
mse_ols = mean_squared_error(y_test, y_pred_ols)
r2_ols  = r2_score(y_test, y_pred_ols)

print("─── OLS ───")
print(f"훈련 RMSE : {np.sqrt(mean_squared_error(y_train, ols.predict(X_train_s))):.3f}")
print(f"시험 RMSE : {np.sqrt(mse_ols):.3f}")
print(f"시험 R²   : {r2_ols:.3f}")
print(f"\nOLS 의 계수 (참값과 비교):")
for j in range(8):
    print(f"  β_{j}: 참값 {beta_true[j]:+.2f}    추정 {beta_ols[j]:+.3f}")
print(f"  ... (X_0 과 X_1 이 강하게 상관 → 두 계수가 서로 출렁임)")



## 2.  Ridge 정규화 경로

$\alpha$ 를 $10^{-3}$ 에서 $10^{3}$ 까지 변화시키며 모든 계수가 어떻게 움직이는지 추적한다. Ridge 에서는 \"모든 계수가 부드럽게 0 으로 수렴\" 하는 모양이 보여야 한다.


In [ ]:
alphas = np.logspace(-3, 3, 60)
ridge_path = np.zeros((len(alphas), p))

for i, a in enumerate(alphas):
    m = Ridge(alpha=a).fit(X_train_s, y_train)
    ridge_path[i] = m.coef_

# 그래프
fig, ax = plt.subplots(figsize=(8, 5))
for j in range(p):
    color = "#1E2761" if beta_true[j] != 0 else "#9CA0AB"
    lw    = 2.2          if beta_true[j] != 0 else 0.8
    alpha_line = 0.95    if beta_true[j] != 0 else 0.5
    ax.plot(alphas, ridge_path[:, j], color=color, linewidth=lw, alpha=alpha_line)

ax.set_xscale("log")
ax.set_xlabel(r"정규화 강도 $\alpha$  (log scale)")
ax.set_ylabel(r"표준화 회귀계수 $\hat\beta_j$")
ax.set_title("Ridge 정규화 경로  —  α 가 커지면 모든 계수가 0 으로 부드럽게 수렴")
ax.axhline(0, color="gray", linewidth=0.5)
ax.grid(alpha=0.3)
# 범례 표시용
ax.plot([], [], color="#1E2761", linewidth=2.2, label=f"의미있는 변수 ({(beta_true!=0).sum()} 개)")
ax.plot([], [], color="#9CA0AB", linewidth=0.8, label=f"잡음 변수 ({(beta_true==0).sum()} 개)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()



**관찰 포인트.** 모든 곡선이 $\alpha$ 가 커짐에 따라 부드럽게 0 으로 다가가지만, 어느 누구도 \"정확히 0\" 으로 떨어지지 않는다. 이것이 Ridge 의 정체성이다.



## 3.  Lasso 정규화 경로

같은 작업을 Lasso 로 한다. 이번에는 어떤 계수가 \"정확히 0 으로 떨어지는 순간\" 이 보여야 한다.


In [ ]:
alphas_l = np.logspace(-3, 1, 60)
lasso_path = np.zeros((len(alphas_l), p))

for i, a in enumerate(alphas_l):
    m = Lasso(alpha=a, max_iter=20000).fit(X_train_s, y_train)
    lasso_path[i] = m.coef_

fig, ax = plt.subplots(figsize=(8, 5))
for j in range(p):
    color = "#E0A11B" if beta_true[j] != 0 else "#9CA0AB"
    lw    = 2.2       if beta_true[j] != 0 else 0.8
    alpha_line = 0.95 if beta_true[j] != 0 else 0.5
    ax.plot(alphas_l, lasso_path[:, j], color=color, linewidth=lw, alpha=alpha_line)

ax.set_xscale("log")
ax.set_xlabel(r"정규화 강도 $\alpha$  (log scale)")
ax.set_ylabel(r"표준화 회귀계수 $\hat\beta_j$")
ax.set_title("Lasso 정규화 경로  —  α 가 커지면 일부 계수가 정확히 0 으로")
ax.axhline(0, color="gray", linewidth=0.5)
ax.grid(alpha=0.3)
ax.plot([], [], color="#E0A11B", linewidth=2.2, label=f"의미있는 변수 ({(beta_true!=0).sum()} 개)")
ax.plot([], [], color="#9CA0AB", linewidth=0.8, label=f"잡음 변수 ({(beta_true==0).sum()} 개)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# 큰 α 에서 0 이 된 계수 수
for a in [0.01, 0.1, 0.5, 1.0]:
    m = Lasso(alpha=a, max_iter=20000).fit(X_train_s, y_train)
    n_zero = int(np.sum(np.abs(m.coef_) < 1e-8))
    print(f"α = {a:5.2f}  →  0 이 된 계수의 수: {n_zero} / {p}")



**관찰 포인트.** $\alpha$ 가 커지면 어떤 곡선은 0 으로 떨어진 후 \"그대로 머무른다\". 이것이 Lasso 가 자동으로 변수 선택을 수행하는 모습이다.



## 4.  Ridge vs Lasso vs ElasticNet — 한 자리 비교

같은 자료에 세 방법을 모두 적용해 계수의 \"모양\" 을 비교한다. (α 값은 \"중간 수준의 정규화\" 가 되도록 임의 선택)


In [ ]:
# 적당한 α 로 세 모델 적합
m_ridge = Ridge(alpha=10.0).fit(X_train_s, y_train)
m_lasso = Lasso(alpha=0.10, max_iter=20000).fit(X_train_s, y_train)
m_enet  = ElasticNet(alpha=0.10, l1_ratio=0.5, max_iter=20000).fit(X_train_s, y_train)

idx = np.arange(p)

fig, ax = plt.subplots(figsize=(11, 5))
w = 0.22
ax.bar(idx - 1.5*w, beta_true,   width=w, label="참값",          color="#1B1F2A")
ax.bar(idx - 0.5*w, m_ridge.coef_, width=w, label="Ridge",      color="#1E2761")
ax.bar(idx + 0.5*w, m_lasso.coef_, width=w, label="Lasso",      color="#E0A11B")
ax.bar(idx + 1.5*w, m_enet.coef_,  width=w, label="ElasticNet", color="#6D2E46")
ax.axhline(0, color="gray", linewidth=0.6)
ax.set_xticks(idx)
ax.set_xticklabels([f"β{i}" for i in idx], fontsize=8)
ax.set_ylabel("표준화 회귀계수")
ax.set_title("세 정규화 방법의 계수 패턴  ·  잡음 변수(β=0)들의 모양에 주목")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# 0이 된 계수 비교
for name, m in [("Ridge", m_ridge), ("Lasso", m_lasso), ("ElasticNet", m_enet)]:
    n_zero = int(np.sum(np.abs(m.coef_) < 1e-8))
    print(f"{name:11s} → 0 이 된 계수: {n_zero} / {p}")



**관찰 포인트.** Ridge 는 어떤 계수도 0 으로 두지 않지만, Lasso 와 ElasticNet 은 다수의 잡음 변수의 계수를 정확히 0 으로 처리한다. ElasticNet 은 Lasso 보다 살짝 부드럽다 (L2 성분 덕분).



## 5.  교차검증으로 최적 $\alpha$ 선택 → 시험 자료 성능

$\alpha$ 를 임의로 두지 않고 5-fold 교차검증으로 자동 선택한다. 네 방법의 시험 자료 RMSE 를 비교한다.


In [ ]:
# 모형 정의
models = {
    "OLS":        LinearRegression(),
    "Ridge (CV)": RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5),
    "Lasso (CV)": LassoCV(alphas=np.logspace(-3, 1, 50), cv=5, max_iter=20000, random_state=42),
    "ElasticNet (CV)": ElasticNetCV(
        alphas=np.logspace(-3, 1, 50),
        l1_ratio=[0.2, 0.5, 0.7, 0.9],
        cv=5, max_iter=20000, random_state=42),
}

results = []
for name, m in models.items():
    m.fit(X_train_s, y_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, m.predict(X_train_s)))
    rmse_test  = np.sqrt(mean_squared_error(y_test,  m.predict(X_test_s)))
    r2_test    = r2_score(y_test, m.predict(X_test_s))
    n_zero     = int(np.sum(np.abs(m.coef_) < 1e-8))
    alpha_sel  = getattr(m, "alpha_", None)
    l1r_sel    = getattr(m, "l1_ratio_", None)
    results.append((name, rmse_train, rmse_test, r2_test, n_zero, alpha_sel, l1r_sel))

# 표 출력
print(f"{'모형':<20s}  {'훈련 RMSE':>10s}  {'시험 RMSE':>10s}  {'시험 R²':>8s}  {'0 계수':>6s}   선택된 α / ρ")
print("─" * 95)
for name, rt, re_, r2, nz, a, lr in results:
    a_str  = f"α={a:.3f}" if a is not None else "—"
    lr_str = f", ρ={lr:.2f}" if lr is not None else ""
    print(f"{name:<20s}  {rt:>10.3f}  {re_:>10.3f}  {r2:>8.3f}  {nz:>6d}   {a_str}{lr_str}")



**관찰 포인트.**

- 훈련 RMSE 가 가장 작은 모형이 시험 RMSE 도 가장 작은 모형은 아니다 — 그것이 바로 \"과적합\".  
- Ridge / Lasso / ElasticNet 모두 OLS 보다 시험 RMSE 가 작은 것이 일반적이다 (자료 분할에 따라 다름).  
- Lasso 와 ElasticNet 은 다수의 잡음 변수를 정확히 0 으로 만들어 모형 해석을 단순화한다.



## 6.  편향-분산 트레이드오프 — α 가 변하면 RMSE 가 \"U자\" 를 그린다

같은 Ridge 모형을 $\alpha$ 만 바꿔 가며 훈련·시험 RMSE 를 그려보면, 시험 RMSE 가 어떤 $\alpha$ 에서 최저가 되는 \"U자 곡선\" 을 그린다. 이것이 편향-분산 트레이드오프의 시각적 증거이다.


In [ ]:
alphas_curve = np.logspace(-3, 3, 40)
rmse_train_list, rmse_test_list = [], []

for a in alphas_curve:
    m = Ridge(alpha=a).fit(X_train_s, y_train)
    rmse_train_list.append(np.sqrt(mean_squared_error(y_train, m.predict(X_train_s))))
    rmse_test_list.append(np.sqrt(mean_squared_error(y_test,  m.predict(X_test_s))))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alphas_curve, rmse_train_list, color="#1E2761", linewidth=2,
        label="훈련 RMSE", marker="o", markersize=4)
ax.plot(alphas_curve, rmse_test_list,  color="#E0A11B", linewidth=2,
        label="시험 RMSE", marker="s", markersize=4)
ax.axvline(alphas_curve[np.argmin(rmse_test_list)], color="#6D2E46", linestyle="--",
           label=f"최저 시험 RMSE 의 α = {alphas_curve[np.argmin(rmse_test_list)]:.3f}")
ax.set_xscale("log")
ax.set_xlabel(r"$\alpha$  (log scale)")
ax.set_ylabel("RMSE")
ax.set_title("Ridge 의 편향-분산 트레이드오프  ·  시험 RMSE 는 U자")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



**관찰 포인트.**

- $\alpha$ 가 너무 작으면 (좌측 끝) — 거의 OLS, 시험 RMSE 가 크다 (과적합 = 분산이 큼).
- $\alpha$ 가 너무 크면 (우측 끝) — 모든 계수가 0 에 가까워져 시험 RMSE 가 다시 커진다 (과소적합 = 편향이 큼).
- 가운데 어딘가의 $\alpha$ 가 가장 좋다 — 그 \"가운데\" 를 자동으로 찾는 것이 교차검증의 역할.



## 7.  더 큰 자료에 적용 — `sklearn.datasets.make_regression`

수업에서 자주 쓰던 보스턴 주택가격 자료는 sklearn 에서 deprecated 되었으므로, 비슷한 구조의 합성 회귀 자료를 사용한다.


In [ ]:
from sklearn.datasets import make_regression

X_big, y_big, coef_big = make_regression(
    n_samples=500, n_features=50, n_informative=10,
    noise=10.0, coef=True, random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(X_big, y_big, test_size=0.3, random_state=0)

sc = StandardScaler().fit(X_tr)
X_tr_s = sc.transform(X_tr); X_te_s = sc.transform(X_te)

models_big = {
    "OLS":           LinearRegression(),
    "Ridge CV":      RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5),
    "Lasso CV":      LassoCV(alphas=np.logspace(-3, 1, 50), cv=5, max_iter=30000, random_state=0),
    "ElasticNet CV": ElasticNetCV(alphas=np.logspace(-3, 1, 50),
                                  l1_ratio=[0.2, 0.5, 0.8],
                                  cv=5, max_iter=30000, random_state=0),
}

print(f"{'모형':<16s}  {'시험 RMSE':>10s}  {'시험 R²':>8s}  {'0 계수':>6s}")
print("─" * 50)
for name, m in models_big.items():
    m.fit(X_tr_s, y_tr)
    rmse = np.sqrt(mean_squared_error(y_te, m.predict(X_te_s)))
    r2   = r2_score(y_te, m.predict(X_te_s))
    nz   = int(np.sum(np.abs(m.coef_) < 1e-8))
    print(f"{name:<16s}  {rmse:>10.3f}  {r2:>8.3f}  {nz:>6d}")



## 8.  통합 한 줄

| 회귀 | 손실함수 | 해를 구하는 도구 | 변수 선택 |
|---|---|---|---|
| OLS | $\|y - X\beta\|^2$ | 행렬 미분 = 0 | 안 함 |
| Ridge | $\|y - X\beta\|^2 + \alpha\|\beta\|_2^2$ | 행렬 미분 = 0 | 안 함 (모두 작아짐) |
| Lasso | $\|y - X\beta\|^2 + \alpha\|\beta\|_1$ | 좌표하강법 | 자동으로 함 |
| ElasticNet | Ridge + Lasso | 좌표하강법 | 자동으로 함 (그룹 효과) |

모든 회귀가 \"손실함수를 정의하고 그것을 최소화하는 $\beta$ 를 찾는다\" 라는 한 사고방식 위에 있다. 미분 가능 여부에 따라 \"편미분 = 0\" 으로 직접 풀거나 (OLS, Ridge), 반복 알고리즘이 필요할 뿐이다 (Lasso, ElasticNet).



## 9.  연습 문제

1. 본 노트북의 합성 자료에서 잡음 표준편차를 1 → 3 으로 키우면, OLS 와 Ridge 의 시험 RMSE 차이는 어떻게 변하는가? 직접 실험하라.

2. `Lasso(alpha=0.0)` 가 \"이론적으로 OLS 와 같지만 실제로 같지 않을 수 있는\" 이유를 sklearn 문서에서 찾아 적으라.

3. 표준화를 **하지 않고** Ridge 를 적용해 보라. 단위가 큰 변수의 계수가 자동으로 작아짐을 확인하라. 이것이 왜 정규화 직전에 표준화가 필수인지의 실증이다.

4. `make_regression` 으로 $n = 50$, $p = 100$ 자료를 만들어 OLS · Ridge · Lasso 의 시험 RMSE 를 비교하라. OLS 가 실제로 풀리는지부터 확인한다 ($X^{\mathsf T}X$ 가 가역이 아닐 수 있다).
